# 30 septembre — Recherche dichotomique récursive

[Lire le cours et les preuves détaillées](note.md) · [Énoncé du TD 2](../23.09/TD2_original.pdf)

**Repère :** la photo porte sur la terminaison et la correction de la recherche dichotomique. C'est le même thème que le TD 2, mais cet algorithme ne figure pas dans les deux pages de l'énoncé disponible : aucun numéro d'exercice n'est attribué ici.

Le premier code reprend la partie lisible de la projection. Les jeux de données, les explications et la version itérative sont des compléments de révision. Exécuter les cellules dans l'ordre.

## 1. Le contrat de la fonction

Le tableau est trié dans l'ordre croissant au sens large et reste inchangé pendant la recherche. On travaille ici avec des entiers.

- Résultat : un indice `i` tel que `tab[i] == val`, ou `-1` si la valeur est absente.
- Bornes : `g` et `d` sont incluses ; `g > d` signifie que la zone est vide.
- En présence de doublons, n'importe quel indice d'occurrence correct convient.
- Le `print(g, d)` sert à observer les appels ; il ne change pas le résultat.

In [1]:
def recherche_dic(tab, g, d, val):
    print(g, d)
    if g > d:
        return -1
    else:
        m = (g + d) // 2
        if tab[m] == val:
            return m
        elif tab[m] > val:
            return recherche_dic(tab, g, m - 1, val)
        else:  # tab[m] < val
            return recherche_dic(tab, m + 1, d, val)


def recherche(tab, val):
    return recherche_dic(tab, 0, len(tab) - 1, val)


## 2. Valeur présente : suivre la descente et les retours

Pour chercher `16`, on examine successivement les indices 3, 5, puis 4. Le dernier appel renvoie l'indice 4, que les appels précédents transmettent grâce à `return`.

In [2]:
tab = [2, 5, 8, 12, 16, 23, 38]
indice = recherche(tab, 16)
print("Indice trouvé :", indice)
if indice != -1:
    print("Valeur à cet indice :", tab[indice])


0 6
4 6
4 4
Indice trouvé : 4
Valeur à cet indice : 16


## 3. Valeur absente et tableau vide

En cherchant `10`, les bornes finissent à `g = 3`, `d = 2`. La zone vide prouve l'absence, grâce à la propriété conservée pendant la recherche.

Sur `[]`, le premier appel porte directement sur les bornes `0, -1` : aucun accès à une case n'a lieu. Toujours tester le résultat avant de l'utiliser comme indice : en Python, `-1` désigne aussi la dernière case d'une liste non vide.

In [3]:
print("Recherche de 10 :")
assert recherche(tab, 10) == -1

print("Recherche dans un tableau vide :")
assert recherche([], 10) == -1

print("Recherche dans un singleton :")
assert recherche([10], 10) == 0
assert recherche([10], 9) == -1


Recherche de 10 :
0 6
0 2
2 2
3 2
Recherche dans un tableau vide :
0 -1
Recherche dans un singleton :
0 0
0 0
0 -1


## 4. Les doublons et la précondition de tri

Le code ne promet pas la première occurrence : avec `[1, 2, 2, 2, 3]`, le milieu est déjà un `2`, d'indice 2.

Sur un tableau non trié, le programme peut terminer tout en donnant une mauvaise réponse : la terminaison et la correction sont deux propriétés distinctes.

In [4]:
doublons = [1, 2, 2, 2, 3]
i = recherche(doublons, 2)
assert i == 2
print("Un indice correct parmi plusieurs :", i)

# Contre-exemple volontaire : la précondition de tri n'est pas respectée.
non_trie = [8, 1, 5]
r = recherche(non_trie, 8)
print("Sur le tableau non trié :", r, "; pourtant 8 est présent.")


0 4
Un indice correct parmi plusieurs : 2
0 2
2 2
3 2
Sur le tableau non trié : -1 ; pourtant 8 est présent.


## 5. Ce qu'il faut démontrer

**Variant du tableau :** `d - g` diminue strictement à chaque appel et reste positif ou nul tant que la zone est non vide. Il vaut 0 sur un singleton. Pour inclure le dernier appel vide dans les naturels, on peut utiliser `d - g + 1` avec les bornes produites par la fonction.

**Propriété conservée :** si la valeur est dans le tableau initial, une occurrence est encore dans `[g, d]`. Le tri justifie l'élimination de la moitié gauche ou droite. Une égalité donne un indice correct ; une zone vide donne l'absence.

La [fiche de cours](note.md) détaille l'initialisation, les deux cas de conservation et les retours. La récursion diminue le nombre de candidats d'environ moitié : temps logarithmique dans le pire cas, et pile de taille logarithmique, hors affichage.

## 6. Complément : version itérative

Cette version n'est pas transcrite de la photo. Elle conserve la même zone candidate et le même raisonnement, en remplaçant les appels récursifs par une boucle. Elle évite la pile de récursion et utilise O(1) espace supplémentaire dans le modèle usuel.

In [5]:
def recherche_iterative(tab, val):
    g = 0
    d = len(tab) - 1
    while g <= d:
        m = (g + d) // 2
        if tab[m] == val:
            return m
        elif tab[m] > val:
            d = m - 1
        else:
            g = m + 1
    return -1


assert recherche_iterative(tab, 16) == 4
assert recherche_iterative(tab, 10) == -1


## 7. Vérifier les cas limites

On compare chaque résultat au contrat : présence réelle de la valeur à l'indice annoncé, ou absence réelle pour `-1`. On ne demande pas le même indice que `list.index`, car celle-ci impose la première occurrence.

Les essais couvrent le vide, les extrémités, les doublons, les nombres négatifs et les valeurs absentes. Les petites listes triées sont produites par combinaisons avec répétitions ; le tableau doit rester inchangé. Les traces du code du professeur sont masquées pendant ces vérifications pour garder la sortie lisible.

In [6]:
from contextlib import redirect_stdout
from io import StringIO
from itertools import combinations_with_replacement


def verifier_resultat(fonction, valeurs, cible):
    avant = valeurs.copy()
    with redirect_stdout(StringIO()):
        resultat = fonction(valeurs, cible)
    assert valeurs == avant, "La recherche ne doit pas modifier le tableau."
    if cible in valeurs:
        assert 0 <= resultat < len(valeurs)
        assert valeurs[resultat] == cible
    else:
        assert resultat == -1


nombre = 0
for taille in range(7):
    for combinaison in combinations_with_replacement(range(-2, 3), taille):
        valeurs = list(combinaison)
        for cible in range(-3, 4):
            for fonction in (recherche, recherche_iterative):
                verifier_resultat(fonction, valeurs, cible)
                nombre += 1

print(nombre, "recherches vérifiées sur de petites listes triées.")


6468 recherches vérifiées sur de petites listes triées.


## 8. S'entraîner sans regarder la correction

1. Prédire les bornes affichées en cherchant `38`, puis `0`, dans le tableau de l'exemple.
2. Expliquer ce qui se passerait avec `g >= d` à la place de `g > d`.
3. Expliquer pourquoi garder `m` dans la zone de droite avec `g = m` pourrait empêcher la progression.
4. Rédiger le cas de conservation lorsque `tab[m] > val`.

**Éléments de correction :**

1. Pour `38` : `(0, 6)`, `(4, 6)`, `(6, 6)`, résultat 6. Pour `0` : `(0, 6)`, `(0, 2)`, `(0, 0)`, `(0, -1)`, résultat `-1`.
2. Une zone d'une seule case serait déclarée vide, alors qu'elle peut contenir la valeur cherchée.
3. Sur une zone réduite à une case de valeur trop petite, le milieu et les bornes pourraient rester identiques.
4. Pour tout indice `i >= m`, le tri impose `tab[i] >= tab[m] > val`. La valeur cherchée ne peut donc être que dans `[g, m - 1]` si elle existe.

La suite de la séance pourra être ajoutée après cette cellule.